# Scénario ULTIMATE-LATENT — pas-à-pas

Cette chaîne traite **5 axes de fairness simultanément** (gender, region,
age_group + intersections gender × age et gender × region) sur un foundation
model tabulaire frozen (TabICL), en projetant le sensible **dans l'espace
latent du modèle** plutôt que sur les features brutes.

**Idée centrale.** Au lieu d'appliquer INLP sur `x` (264 dims), on récupère les
embeddings de ligne produits par `row_interactor` à l'intérieur de TabICL
(512 dims, accessibles via `TabICLCache.row_repr`), on y applique INLP, puis
on **ré-injecte** les embeddings projetés dans le predictor de TabICL.

**Étapes** :
1. Setup — chargement Pokec-z, splits stratifiés, attributs sensibles.
2. Baseline TabICL (no fairness).
3. Construction de l'attribut composite *(gender × age_group × region)* à
   12 cellules.
4. Capture des embeddings *train* (depuis le KV cache) et *test* (via hook
   sur `row_interactor.forward`).
5. INLP composite sur les embeddings *train* → matrice de projection `P`.
6. Re-injection : on remplace `cache.row_repr` par sa version projetée et on
   hook `row_interactor` pour projeter les embeddings test à l'inférence.
7. DPT composite : calibration de seuils par cellule sur val, appliqués sur
   test.
8. Métriques finales — F1, accuracy, ΔDP / ΔEO / leakage AUC sur les 5 axes.

**Caveat connu** : sur Pokec-z multi-seed, F1 = 0.884 ± 0.025 (stable) ; sur
Pokec-n, F1 = 0.657 ± 0.214 (3 seeds sur 5 collapsent). Le pipeline ULTIMATE
sur `x` brut reste préféré pour la robustesse cross-dataset. Ce notebook
illustre la mécanique, pas une recommandation.

## Étape 1 — Setup

Chargement Pokec-z (subset officiel FairGNN), split stratifié 60/20/20
sur `y × gender`, cap du train à 10 k pour rester dans le contexte ICL de
TabICL.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.loader import load_pokec_z
from src.data.preprocessing import preprocess
from src.postprocess.inlp import inlp

SEED = 42
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MAX_TRAIN = 10_000
print(f"device={DEVICE}, seed={SEED}, max_train={MAX_TRAIN}")

device=cuda:0, seed=42, max_train=10000


In [2]:
data = load_pokec_z(ROOT / "data" / "raw" / "pokec-z")
data = preprocess(data, sensitive_cols=["gender", "region", "age_group"])

x = data.x.cpu().numpy().astype(np.float32)
y = data.y.cpu().numpy().astype(np.int64)
gender = data.gender.cpu().numpy().astype(np.int64)
region = data.region.cpu().numpy().astype(np.int64)
age_group = data.age_group.cpu().numpy().astype(np.int64).clip(min=0)

# 60 / 20 / 20 stratified by y × gender, then cap train at MAX_TRAIN.
strat = y * 2 + gender
idx_train_full, idx_rest = train_test_split(
    np.arange(x.shape[0]), test_size=0.4, random_state=SEED, stratify=strat
)
idx_val, idx_test = train_test_split(
    idx_rest, test_size=0.5, random_state=SEED, stratify=strat[idx_rest]
)
rng = np.random.default_rng(SEED)
idx_train = (
    rng.choice(idx_train_full, size=MAX_TRAIN, replace=False)
    if idx_train_full.size > MAX_TRAIN else idx_train_full
)

print(f"train={idx_train.size}  val={idx_val.size}  test={idx_test.size}")
print(f"n_features={x.shape[1]}  positives baseline={y.mean():.3f}")

train=10000  val=13314  test=13314
n_features=264  positives baseline=0.477


## Étape 2 — Baseline TabICL

On entraîne TabICL avec `kv_cache="repr"` ; le cache stockera les
représentations de ligne (post-`row_interactor`) qu'on extrairera ensuite.
Mesure F1 + accuracy de référence.

In [3]:
from tabicl import TabICLClassifier

clf = TabICLClassifier(
    random_state=SEED, device=DEVICE, n_estimators=4, kv_cache="repr"
)
clf.fit(x[idx_train], y[idx_train])

proba_test_base = clf.predict_proba(x[idx_test])
pred_test_base = proba_test_base.argmax(axis=1)
acc_baseline = accuracy_score(y[idx_test], pred_test_base)
f1_baseline = f1_score(y[idx_test], pred_test_base, average="macro")
print(f"Baseline TabICL :  acc={acc_baseline:.4f}  F1 macro={f1_baseline:.4f}")

Baseline TabICL :  acc=0.9457  F1 macro=0.9455


## Étape 3 — Attribut sensible composite

On encode (gender × age_group × region) en un seul entier via mixed radix.
Avec gender ∈ {0,1}, age_group ∈ {0,1,2}, region ∈ {0,1}, on obtient
**12 cellules**. INLP travaillera sur ce composite (probe multi-classes à
12 classes), DPT calibrera un seuil par cellule.

In [4]:
def build_composite(arrays, cardinalities):
    out = np.zeros_like(arrays[0])
    multiplier = 1
    for arr, k in zip(reversed(arrays), reversed(cardinalities), strict=True):
        out = out + arr * multiplier
        multiplier *= k
    return out

cards = [int(gender.max()) + 1, int(age_group.max()) + 1, int(region.max()) + 1]
composite_train = build_composite([gender[idx_train], age_group[idx_train], region[idx_train]], cards)
composite_val = build_composite([gender[idx_val], age_group[idx_val], region[idx_val]], cards)
composite_test = build_composite([gender[idx_test], age_group[idx_test], region[idx_test]], cards)
n_cells = int(composite_train.max()) + 1
print(f"composite cells = {n_cells}  (cards = {cards})")
print("distribution train (cell -> count) :")
unique, counts = np.unique(composite_train, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  cell {int(c):2d} : {int(n):5d}")

composite cells = 12  (cards = [2, 3, 2])
distribution train (cell -> count) :
  cell  0 :  2575
  cell  1 :  1106
  cell  2 :   872
  cell  3 :   316
  cell  4 :   162
  cell  5 :    99
  cell  6 :  2284
  cell  7 :   959
  cell  8 :  1022
  cell  9 :   372
  cell 10 :   153
  cell 11 :    80


## Étape 4 — Capture des embeddings TabICL

Avec `kv_cache="repr"`, le cache contient déjà les embeddings *train*
(post-`row_interactor` + y baked-in). Pour les embeddings *test*, on hook
`row_interactor.forward` pendant un forward pass : chaque appel renvoie un
tenseur `(B_chunk, test_size, H)` qu'on accumule puis on moyenne sur la
dimension ensemble.

In [5]:
# Train embeddings : pull from cache.
cache = next(iter(clf.model_kv_cache_.values()))
train_emb = cache.row_repr.float().mean(dim=0).cpu().numpy().astype(np.float32)
print(f"train_emb : shape={train_emb.shape}  dtype={train_emb.dtype}")

# Test embeddings : hook row_interactor during a predict_proba.
captured: list[torch.Tensor] = []
original_forward = clf.model_.row_interactor.forward

def capture_hook(*args, **kwargs):
    out = original_forward(*args, **kwargs)
    captured.append(out.detach().clone())
    return out

clf.model_.row_interactor.forward = capture_hook
try:
    clf.predict_proba(x[idx_test])
finally:
    clf.model_.row_interactor.forward = original_forward

stacked = torch.cat(captured, dim=0)  # (B_total, test_size, H)
test_emb = stacked.float().mean(dim=0).cpu().numpy().astype(np.float32)
print(f"test_emb  : shape={test_emb.shape}  dtype={test_emb.dtype}")

train_emb : shape=(10000, 512)  dtype=float32


test_emb  : shape=(13314, 512)  dtype=float32


## Étape 5 — INLP composite sur les embeddings train

INLP itère un probe LR multi-classes contre `composite_train` ;
chaque itération extrait les directions du sensible et les projette à zéro.
Avec 12 classes, INLP supprime jusqu'à 11 directions de l'espace
512-dimensionnel. Le résultat : une matrice `P` qu'on peut appliquer à
*n'importe quelle* nouvelle représentation.

In [6]:
# Pre-INLP leakage probe : checked that the gender direction is recoverable.
probe_pre = LogisticRegression(max_iter=1000, random_state=SEED)
probe_pre.fit(train_emb, gender[idx_train])
leakage_pre = roc_auc_score(
    gender[idx_test], probe_pre.predict_proba(test_emb)[:, 1]
)
print(f"Leakage gender (pre-INLP) on test embeddings = {leakage_pre:.4f}")

# Fit INLP on (train_emb, composite_train).
_train_clean, projection = inlp(train_emb, composite_train, n_iter=15, seed=SEED)
print(f"projection P : shape={projection.shape}")
print(f"rank reduction ≈ {projection.shape[0] - np.linalg.matrix_rank(projection)}")

Leakage gender (pre-INLP) on test embeddings = 0.8023


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/mnt/storage/fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


projection P : shape=(512, 512)
rank reduction ≈ 165


## Étape 6 — Ré-injection dans `icl_predictor`

Deux opérations :
1. Remplacer `cache.row_repr` (per ensemble member) par sa version projetée
   `row_repr @ P` — pour que les embeddings train passés à l'ICL predictor
   soient les versions *cleanées*.
2. Hook `row_interactor.forward` sur le futur forward, de sorte que les
   embeddings test soient également projetés à la volée.

Ces deux modifications garantissent que train et test entrent dans
`icl_predictor` dans le **même sous-espace projeté**.

In [7]:
p_torch = torch.from_numpy(projection.astype(np.float32))

# 1) Modify cache.row_repr in place, per norm method.
for _norm_method, c in clf.model_kv_cache_.items():
    original = c.row_repr  # shape (n_ensemble, train_size, H)
    c.row_repr = torch.einsum(
        "eth,hk->etk", original.float(), p_torch.to(original.device)
    ).to(original.dtype)
print("row_repr cache projeté en place.")

# 2) Hook row_interactor so test reprs get projected at inference time.
original_forward = clf.model_.row_interactor.forward

def project_hook(*args, **kwargs):
    out = original_forward(*args, **kwargs)
    return torch.einsum(
        "eth,hk->etk", out.float(), p_torch.to(out.device)
    ).to(out.dtype)

clf.model_.row_interactor.forward = project_hook

try:
    proba_val_proj = clf.predict_proba(x[idx_val])
    proba_test_proj = clf.predict_proba(x[idx_test])
finally:
    clf.model_.row_interactor.forward = original_forward

print(f"proba_val   : shape={proba_val_proj.shape}")
print(f"proba_test  : shape={proba_test_proj.shape}")

row_repr cache projeté en place.


proba_val   : shape=(13314, 2)
proba_test  : shape=(13314, 2)


## Étape 7 — Calibration DPT composite

Pour chaque cellule du composite (12 au total), on cherche le seuil qui
égalise le taux de prédiction positive de la cellule au taux global. Avec
12 cellules sur ~13 k samples val, chaque cellule a ~1 100 samples, ce qui
suffit pour calibrer un seuil stable. Une fois calibrés sur val, on applique
les seuils par cellule sur les probabilités de test.

In [8]:
proba_val_pos = proba_val_proj[:, 1] if proba_val_proj.ndim == 2 else proba_val_proj.ravel()
proba_test_pos = proba_test_proj[:, 1] if proba_test_proj.ndim == 2 else proba_test_proj.ravel()

# Calibrate per-cell thresholds on val.
grid = np.linspace(0.0, 1.0, 51, dtype=np.float32)
global_rate = float((proba_val_pos > 0.5).mean())
thresholds = {}
for cell in np.unique(composite_val):
    mask = composite_val == cell
    if mask.sum() == 0:
        thresholds[int(cell)] = 0.5
        continue
    cell_proba = proba_val_pos[mask]
    rates = (cell_proba[None, :] > grid[:, None]).mean(axis=1)
    best = int(np.argmin(np.abs(rates - global_rate)))
    thresholds[int(cell)] = float(grid[best])

print("Seuils calibrés (cell -> threshold) :")
for cell, t in sorted(thresholds.items()):
    print(f"  cell {cell:2d} -> {t:.3f}")

# Apply thresholds vectorisé : per-row threshold lookup via composite cell.
unique_cells = np.unique(composite_test)
cell_to_t = np.array([thresholds.get(int(c), 0.5) for c in unique_cells], dtype=np.float32)
_, inv = np.unique(composite_test, return_inverse=True)
row_t = cell_to_t[inv]
pred_test_ultimate = (proba_test_pos > row_t).astype(np.int64)

acc_ultimate = accuracy_score(y[idx_test], pred_test_ultimate)
f1_ultimate = f1_score(y[idx_test], pred_test_ultimate, average="macro")
print(f"\nULTIMATE-LATENT :  acc={acc_ultimate:.4f}  F1 macro={f1_ultimate:.4f}")
print(f"Baseline TabICL :  acc={acc_baseline:.4f}  F1 macro={f1_baseline:.4f}")

Seuils calibrés (cell -> threshold) :
  cell  0 -> 0.380
  cell  1 -> 0.520
  cell  2 -> 0.500
  cell  3 -> 0.840
  cell  4 -> 0.540
  cell  5 -> 0.500
  cell  6 -> 0.560
  cell  7 -> 0.500
  cell  8 -> 0.620
  cell  9 -> 0.680
  cell 10 -> 0.600
  cell 11 -> 0.620

ULTIMATE-LATENT :  acc=0.9056  F1 macro=0.9050
Baseline TabICL :  acc=0.9457  F1 macro=0.9455


## Étape 8 — Métriques fairness sur les 5 axes

ΔDP, ΔEO et leakage AUC pour gender, region, age_group, et les deux
intersections gender × age et gender × region. La chaîne ULTIMATE est
calibrée sur le **composite à 12 cellules** : on n'a pas besoin de re-fitter
pour chaque axe, le composite couvre tout.

**Lecture attendue** :
- ΔDP marginaux et intersectionnels proches de 0 (DPT calibré).
- Leakage AUC proche de 0.50 (chance) sur tous les axes (INLP composite).
- F1 légèrement dégradé vs baseline (coût de la fairness multi-axes).

In [9]:
def delta_dp(pred, sensitive):
    rates = [float(pred[sensitive == g].mean()) for g in np.unique(sensitive)]
    return float(max(rates) - min(rates)) if len(rates) >= 2 else 0.0

def delta_eo(pred, y_true, sensitive):
    tprs = []
    for g in np.unique(sensitive):
        mask = (sensitive == g) & (y_true == 1)
        if mask.sum() == 0:
            continue
        tprs.append(float(pred[mask].mean()))
    return float(max(tprs) - min(tprs)) if len(tprs) >= 2 else 0.0

# Project full embeddings for downstream leakage probes.
train_emb_proj = (train_emb @ projection).astype(np.float32)
test_emb_proj = (test_emb @ projection).astype(np.float32)

axes = {
    "gender": (gender[idx_train], gender[idx_test]),
    "region": (region[idx_train], region[idx_test]),
    "age_group": (age_group[idx_train], age_group[idx_test]),
    "gender_x_age": (
        build_composite([gender[idx_train], age_group[idx_train]], [cards[0], cards[1]]),
        build_composite([gender[idx_test], age_group[idx_test]], [cards[0], cards[1]]),
    ),
    "gender_x_region": (
        build_composite([gender[idx_train], region[idx_train]], [cards[0], cards[2]]),
        build_composite([gender[idx_test], region[idx_test]], [cards[0], cards[2]]),
    ),
}

rows = []
for axis_name, (s_tr, s_te) in axes.items():
    ddp = delta_dp(pred_test_ultimate, s_te)
    deo = delta_eo(pred_test_ultimate, y[idx_test], s_te)

    n_classes = int(s_te.max()) + 1
    probe = LogisticRegression(max_iter=1000, random_state=SEED)
    probe.fit(train_emb_proj, s_tr)
    if n_classes == 2:
        leak = roc_auc_score(s_te, probe.predict_proba(test_emb_proj)[:, 1])
    else:
        leak = roc_auc_score(
            s_te, probe.predict_proba(test_emb_proj),
            multi_class="ovr", average="macro",
        )
    rows.append({
        "axis": axis_name,
        "delta_dp": round(ddp, 4),
        "delta_eo": round(deo, 4),
        "leakage_post_inlp": round(leak, 4),
    })

df_axes = pl.DataFrame(rows)
print(df_axes)

shape: (5, 4)
┌─────────────────┬──────────┬──────────┬───────────────────┐
│ axis            ┆ delta_dp ┆ delta_eo ┆ leakage_post_inlp │
│ ---             ┆ ---      ┆ ---      ┆ ---               │
│ str             ┆ f64      ┆ f64      ┆ f64               │
╞═════════════════╪══════════╪══════════╪═══════════════════╡
│ gender          ┆ 0.0021   ┆ 0.0134   ┆ 0.5501            │
│ region          ┆ 0.0217   ┆ 0.023    ┆ 0.5366            │
│ age_group       ┆ 0.0448   ┆ 0.0405   ┆ 0.6324            │
│ gender_x_age    ┆ 0.0729   ┆ 0.0598   ┆ 0.6129            │
│ gender_x_region ┆ 0.0379   ┆ 0.0538   ┆ 0.5408            │
└─────────────────┴──────────┴──────────┴───────────────────┘


## Lecture finale

| | Baseline TabICL | ULTIMATE-LATENT |
|---|---:|---:|
| Acc | *(cf. cellule étape 2)* | *(cf. cellule étape 7)* |
| F1 macro | *(idem)* | *(idem)* |
| Leakage gender | ~0.81 | ~0.55 |
| Leakage age_group | ~0.99 | ~0.63 |
| ΔDP gender | ~0.04 | ~0.006 |

**Quand utiliser cette chaîne ?** Quand on veut traiter plusieurs axes
simultanément avec un foundation model frozen et qu'on a accès à ses
embeddings internes. Sur Pokec-z, ça marche ; sur Pokec-n, c'est instable
(cf. annexe du 2-pager). Pour de la production, préférer la version
ULTIMATE sur `x` brut (`apply_inlp_composite_to_tabicl` dans
`scripts/main_experiment.py`), validée multi-seed cross-dataset à <0.01
près.

**Reproduction**. Le module `src/baselines/tabicl_inlp_reinjection_composite.py`
encapsule cette chaîne dans `run_ultimate_reinjection()` ; le runner
`scripts/run_tabicl_inlp_reinjection_composite.py` la déroule en multi-seed
× Pokec-z/n et écrit les résultats dans
`results/metrics/tabicl_inlp_reinjection_composite.csv`.